# Sampling-scheme benchmark protocol

**Research question:** On the 1D viscous Burgers equation, does an adaptive residual-based resampling strategy produce more accurate PINN solutions than static or periodically re-sampled Latin Hypercube Sampling (LHS)?

**Test problem:** 1D Burgers equation on $x \in [-1, 1]$, $t \in [0, 1]$ with $\nu = 0.01/\pi$ and sinusoidal initial condition $u(x,0) = -\sin(\pi x)$, zero Dirichlet boundaries. Reference solution loaded from a pre-computed finite-difference file.

**Schemes compared:**
- `r3`: initial LHS + boundary LHS refresh every 50 epochs + R3 residual-based interior resampling every epoch.
- `random`: initial LHS only; no resampling during training (static baseline).
- `randomr`: full LHS re-sample every 50 epochs (periodic non-adaptive baseline).

**Justification:** Static LHS is a common default; periodic LHS tests whether resampling alone helps; R3 tests whether residual-guided adaptive resampling adds value beyond plain resampling.

**Controls:** identical network ($\text{width}=16$, $\text{length}=4$), optimizer (Adam, lr=0.004), epochs (5000), and sample budget (1000 boundary + 500 + 500 initial/boundary + 4000 interior) for every method. A fresh model is instantiated for each method and each independent run to avoid weight leakage.

**Metrics:** relative L2 and L1 error against the reference solution, averaged over repeated runs.

**Seeding:** one global seed is fixed at the start of the notebook. Each independent run uses a deterministic run seed (`69 + i`), and the same run seed is re-applied immediately before creating each method's model so all three methods begin each run with identical initial network parameters.

In [ ]:
import deepflow as df
print("Deepflow is runned on:", df.device) # to change to cpu use df.device = 'cpu'
df.manual_seed(69) # for reproducibility

sampling_methods = ['r3', 'random', 'randomr']
obj_dict = {method:{} for method in sampling_methods}
for method in sampling_methods:
    obj_dict[method]["L1_list"] = []
    obj_dict[method]["L2_list"] = []

for i in range(10):
    run_seed = 69 + i

    # Define Geometry and Computational Domain
    area = df.geometry.rectangle([-1, 1], [0, 1])
    line_ic = df.geometry.line_horizontal(y=0, range_x=[-1,1])
    line_bc1 = df.geometry.line_vertical(x=-1, range_y=[0,1])
    line_bc2 = df.geometry.line_vertical(x=1, range_y=[0,1])
    domain = df.domain(area.area_list, line_ic, line_bc1, line_bc2)

    # Define PDE
    from torch import sin, pi
    domain.area_list[0].define_pde(df.pde.BurgersEquation1D(nu=0.01/pi))
    domain.bound_list[0].define_bc({'u':['x', lambda x: -sin(pi * x)]})
    domain.bound_list[1].define_bc({'u': 0})
    domain.bound_list[2].define_bc({'u': 0})

    def do_random(epoch, model):
        pass
    obj_dict["random"]["do"] = do_random
    obj_dict["random"]["lr"] = 0.004

    def do_randomr(epoch, model):
        if epoch % 50 == 0 and epoch > 0:
            domain.sampling_lhs([1000, 500, 500], [4000])
    obj_dict["randomr"]["do"] = do_randomr
    obj_dict["randomr"]["lr"] = 0.004

    def do_r3(epoch, model):
        if epoch % 50 == 0 and epoch > 0:
            domain.sampling_lhs([1000, 500, 500])
        if epoch > 0:
            domain.sampling_R3([], [4000])
    obj_dict["r3"]["do"] = do_r3
    obj_dict["r3"]["lr"] = 0.004


    for method in sampling_methods:
        df.manual_seed(run_seed)
        model0 = df.PINN(input_vars=['x', 'y'], output_vars=['u'], width=16, length=4)
        domain.sampling_lhs([1000, 500, 500], [4000])
        obj_dict[method]["model"], _ = model0.train_adam(
            calc_loss = df.calc_loss_weighted(domain, bc_weights=1),
            learning_rate=obj_dict[method]["lr"],
            epochs=5000,
            do_between_epochs=obj_dict[method]["do"],
            print_every=1000)

    import numpy as np
    with open('../../EXPERIMENTS/burger_sol/burgers_solution.txt', 'r') as f:
        data = np.loadtxt(f)
    x, y, u = data[:, 0], data[:, 1], data[:, 2]

    import torch
    domain_test = df.custom_data({'x':torch.tensor(x, dtype=torch.float32), 'y':torch.tensor(y, dtype=torch.float32)})

    for method in sampling_methods:
        prediction = obj_dict[method]["prediction"] = domain_test.evaluate(obj_dict[method]["model"])
        prediction['u_sol'] = u
        prediction['u_error'] = np.abs(prediction['u']-prediction['u_sol'])
        obj_dict[method]['L2'] = np.sqrt(np.sum(prediction['u_error']**2)/np.sum(prediction['u_sol']**2))
        obj_dict[method]['L1'] = np.sum(prediction['u_error'])/np.sum(np.abs(prediction['u_sol']))

        print('L2 error for method', method, ':', obj_dict[method]['L2'])
        print('L1 error for method', method, ':', obj_dict[method]['L1'])
        obj_dict[method]['L2_list'].append(obj_dict[method]['L2'])
        obj_dict[method]['L1_list'].append(obj_dict[method]['L1'])

Deepflow is runned on: cuda
Epoch: 1, total_loss: 0.65493, bc_loss: 0.65191, pde_loss: 0.00303
Epoch: 1000, total_loss: 0.15874, bc_loss: 0.10630, pde_loss: 0.05245
Epoch: 2000, total_loss: 0.13188, bc_loss: 0.09220, pde_loss: 0.03968
Epoch: 3000, total_loss: 0.04357, bc_loss: 0.02710, pde_loss: 0.01647
Epoch: 4000, total_loss: 0.01660, bc_loss: 0.00862, pde_loss: 0.00797
Epoch: 5000, total_loss: 0.01604, bc_loss: 0.00368, pde_loss: 0.01236
Epoch: 5000, total_loss: 0.01604, bc_loss: 0.00368, pde_loss: 0.01236
Epoch: 1, total_loss: 0.65493, bc_loss: 0.65191, pde_loss: 0.00303
Epoch: 1000, total_loss: 0.08202, bc_loss: 0.04863, pde_loss: 0.03339
Epoch: 2000, total_loss: 0.03605, bc_loss: 0.02061, pde_loss: 0.01545
Epoch: 3000, total_loss: 0.00458, bc_loss: 0.00154, pde_loss: 0.00304
Epoch: 4000, total_loss: 0.00237, bc_loss: 0.00082, pde_loss: 0.00155
Epoch: 5000, total_loss: 0.00152, bc_loss: 0.00052, pde_loss: 0.00099
Epoch: 5000, total_loss: 0.00152, bc_loss: 0.00052, pde_loss: 0.0009

In [ ]:
for method in sampling_methods:
    print(method + " L2:", np.mean(obj_dict[method]['L2_list']))
    print(method + " L1:", np.mean(obj_dict[method]['L1_list']))

import matplotlib.pyplot as plt
for method in sampling_methods:
    plt.plot(obj_dict[method]['L2_list'], label=method)
plt.xlabel('Epoch')
plt.ylabel('L2 Error')
plt.title('Convergence of Different Sampling Methods')
plt.legend()
plt.show()